[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/14_Laplace_Transforms.ipynb)

# DiveLab

## Notebook 14 — Laplace Transforms: From Differential Equations to the $s$-Domain

**Guiding question:** Why do control engineers transform a dynamical system from time $t$ into a new variable $s$?

We already know how to describe diver dynamics with differential equations and state-space models.

Now we introduce another language:

$$
\boxed{\text{time domain} \longrightarrow \text{Laplace domain}}
$$

The Laplace transform turns differentiation into algebra. That will prepare us for **transfer functions, poles, frequency response, and classical control**.

## Learning objectives

By the end of this notebook, you will be able to:

- explain the Laplace transform in simple words;
- understand the meaning of the complex variable $s$;
- transform common signals;
- see why derivatives become algebraic expressions;
- include initial conditions correctly;
- solve simple differential equations with Laplace transforms;
- connect exponential modes, eigenvalues and poles;
- apply the method to a linearized diver model;
- understand why Laplace transforms are useful for control engineering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

sp.init_printing()

# 1. Why another representation?

Consider a first-order dynamical system:

$$
\dot x(t)+a x(t)=u(t).
$$

In the time domain this is a differential equation.

After a Laplace transform, under zero initial conditions, it becomes:

$$
sX(s)+aX(s)=U(s).
$$

Therefore:

$$
X(s)=\frac{1}{s+a}U(s).
$$

The differential equation has become an **algebraic equation**.

That is the central reason Laplace transforms are so useful.

# 2. What is the Laplace transform?

For a time signal $f(t)$, the one-sided Laplace transform is:

$$
F(s)
=
\mathcal{L}\{f(t)\}
=
\int_0^\infty f(t)e^{-st}\,dt.
$$

We often write:

$$
f(t)\quad \longleftrightarrow \quad F(s).
$$

The transform does not change the physical system.

It gives us a different mathematical representation of the same dynamics.

# 3. What is $s$?

The Laplace variable is complex:

$$
s=\sigma+j\omega.
$$

Therefore:

$$
e^{-st}
=
e^{-\sigma t}e^{-j\omega t}.
$$

This contains two ideas:

- $\sigma$: exponential growth or decay;
- $\omega$: oscillation.

That is why the $s$-plane will later become a natural map of dynamical behavior.

In [ ]:
t = np.linspace(0, 10, 1000)

signals = {
    "decay": np.exp(-0.5*t),
    "oscillation": np.cos(2*t),
    "damped oscillation": np.exp(-0.3*t)*np.cos(2*t)
}

for label, y in signals.items():
    plt.plot(t, y, label=label)

plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title("Growth/decay and oscillation in the time domain")
plt.grid(True)
plt.legend()
plt.show()

# 4. First transform: a constant

Let:

$$
f(t)=1,\qquad t\ge0.
$$

Then:

$$
F(s)
=
\int_0^\infty e^{-st}\,dt
=
\frac{1}{s},
$$

for values of $s$ where the integral converges.

So:

$$
\boxed{1 \longleftrightarrow \frac{1}{s}}
$$

This is also the transform of the unit-step signal.

In [ ]:
t_sym, s_sym = sp.symbols("t s", positive=True)
sp.laplace_transform(1, t_sym, s_sym, noconds=True)

# 5. Exponential signals

For:

$$
f(t)=e^{-at},
$$

the transform is:

$$
F(s)=\frac{1}{s+a}.
$$

So:

$$
\boxed{e^{-at}\longleftrightarrow\frac{1}{s+a}}
$$

Notice the connection:

- time-domain decay rate: $-a$;
- denominator root: $s=-a$.

This will become the idea of a **pole** in Notebook 15.

In [ ]:
a = sp.symbols("a", positive=True)
sp.laplace_transform(sp.exp(-a*t_sym), t_sym, s_sym, noconds=True)

# 6. Sine and cosine

Two important transform pairs are:

$$
\sin(\omega t)
\longleftrightarrow
\frac{\omega}{s^2+\omega^2},
$$

and:

$$
\cos(\omega t)
\longleftrightarrow
\frac{s}{s^2+\omega^2}.
$$

Oscillation in time appears through quadratic factors in $s$.

In [ ]:
omega = sp.symbols("omega", positive=True)

L_sin = sp.laplace_transform(sp.sin(omega*t_sym), t_sym, s_sym, noconds=True)
L_cos = sp.laplace_transform(sp.cos(omega*t_sym), t_sym, s_sym, noconds=True)

print("Laplace{sin(omega t)} =", L_sin)
print("Laplace{cos(omega t)} =", L_cos)

# 7. A small transform table

Some pairs worth recognizing are:

$$
1
\longleftrightarrow
\frac{1}{s}
$$

$$
t
\longleftrightarrow
\frac{1}{s^2}
$$

$$
e^{-at}
\longleftrightarrow
\frac{1}{s+a}
$$

$$
\sin(\omega t)
\longleftrightarrow
\frac{\omega}{s^2+\omega^2}
$$

$$
\cos(\omega t)
\longleftrightarrow
\frac{s}{s^2+\omega^2}.
$$

You do not need to memorize a large table yet. The important goal is to understand the structure.

# 8. The key property: differentiation becomes multiplication by $s$

If:

$$
F(s)=\mathcal{L}\{f(t)\},
$$

then:

$$
\boxed{
\mathcal{L}\{\dot f(t)\}
=
sF(s)-f(0)
}
$$

and:

$$
\boxed{
\mathcal{L}\{\ddot f(t)\}
=
s^2F(s)-sf(0)-\dot f(0)
}
$$

This is the bridge from differential equations to algebra.

## Zero initial conditions

If:

$$
f(0)=0,
$$

then:

$$
\mathcal{L}\{\dot f\}=sF(s).
$$

If also:

$$
\dot f(0)=0,
$$

then:

$$
\mathcal{L}\{\ddot f\}=s^2F(s).
$$

This is why transfer functions are usually defined using **zero initial conditions**.

# 9. Solve a first-order differential equation

Consider:

$$
\dot x(t)+2x(t)=u(t),
$$

with:

$$
x(0)=0
$$

and a unit-step input:

$$
u(t)=1.
$$

Taking Laplace transforms:

$$
sX(s)+2X(s)=\frac{1}{s}.
$$

Therefore:

$$
X(s)
=
\frac{1}{s(s+2)}.
$$

In [ ]:
s = sp.symbols("s")
X = 1 / (s * (s + 2))
sp.apart(X, s)

Partial fractions give:

$$
X(s)
=
\frac{1}{2s}
-
\frac{1}{2(s+2)}.
$$

Taking the inverse Laplace transform:

$$
x(t)
=
\frac12
-
\frac12e^{-2t}.
$$

The algebra in the $s$-domain gives us the time-domain solution.

In [ ]:
t = sp.symbols("t", positive=True)
x_t = sp.inverse_laplace_transform(X, s, t)
sp.simplify(x_t)

In [ ]:
tt = np.linspace(0, 5, 500)
xx = 0.5 * (1 - np.exp(-2*tt))

plt.plot(tt, xx)
plt.axhline(0.5, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("x(t)")
plt.title("Step response of a first-order system")
plt.grid(True)
plt.show()

# 10. What did Laplace actually do?

We started with:

$$
\dot x+2x=u.
$$

The derivative made this a dynamical equation.

Laplace transformed it into:

$$
(s+2)X=U.
$$

Now dynamics are encoded in an algebraic factor:

$$
s+2.
$$

The location:

$$
s=-2
$$

contains information about the natural decay of the system:

$$
e^{-2t}.
$$

This is a profound connection:

$$
\boxed{\text{exponential mode in time} \leftrightarrow \text{location in the }s\text{-plane}}
$$

# 11. Initial conditions matter

Now suppose:

$$
\dot x+2x=0
$$

but:

$$
x(0)=x_0.
$$

Then:

$$
sX(s)-x_0+2X(s)=0.
$$

So:

$$
X(s)=\frac{x_0}{s+2}.
$$

Therefore:

$$
x(t)=x_0e^{-2t}.
$$

The system can move even with no external input because it stores an initial state.

This distinction will be important:

- **zero-input response**: motion caused by initial conditions;
- **zero-state response**: motion caused by an input when initial conditions are zero.

Transfer functions focus on the second case.

# 12. A second-order system

Consider:

$$
\ddot x+3\dot x+2x=u.
$$

With zero initial conditions:

$$
s^2X+3sX+2X=U.
$$

Thus:

$$
(s^2+3s+2)X=U.
$$

Factor the polynomial:

$$
s^2+3s+2=(s+1)(s+2).
$$

The natural dynamics therefore contain modes associated with:

$$
s=-1,\qquad s=-2.
$$

In [ ]:
poly = s**2 + 3*s + 2
sp.factor(poly), sp.solve(poly, s)

# 13. Connect this to eigenvalues

Earlier in DiveLab, state-space models used:

$$
\dot x=Ax.
$$

Natural modes had the form:

$$
e^{\lambda t},
$$

where $\lambda$ is an eigenvalue of $A$.

Now Laplace analysis produces denominator roots in the $s$-plane.

For a linear system, these are deeply connected:

$$
\boxed{\lambda \leftrightarrow s_{\text{pole}}}
$$

So eigenvalues and poles are two views of the same underlying dynamics.

# 14. Return to the diver

Near an equilibrium, suppose the vertical dynamics are approximated by:

$$
\dot z=-v
$$

$$
\dot v=a_z z + b u.
$$

Here $z$ represents a small depth deviation from equilibrium.

Differentiate the first equation:

$$
\ddot z=-\dot v.
$$

Substitute the second equation:

$$
\ddot z=-a_z z-bu.
$$

Therefore:

$$
\boxed{
\ddot z+a_z z=-bu
}
$$

depending on the sign convention and the value of $a_z$.

# 15. Laplace-transform the diver equation

With zero initial conditions:

$$
s^2Z(s)+a_zZ(s)=-bU(s).
$$

Factor $Z(s)$:

$$
(s^2+a_z)Z(s)=-bU(s).
$$

So:

$$
Z(s)
=
\frac{-b}{s^2+a_z}U(s).
$$

We have converted a differential equation for vertical motion into an algebraic input-output relationship.

# 16. The unstable buoyancy case

From our earlier nonlinear model, increasing depth compresses gas and reduces buoyancy.

Around the equilibrium this gives:

$$
a_z<0.
$$

Write:

$$
a_z=-\alpha^2,
\qquad \alpha>0.
$$

Then:

$$
s^2+a_z
=
s^2-\alpha^2
=
(s-\alpha)(s+\alpha).
$$

The two characteristic locations are:

$$
s=+\alpha,
\qquad
s=-\alpha.
$$

One lies in the right half-plane.

That corresponds to the growing mode:

$$
e^{+\alpha t}.
$$

This is the same **saddle-point instability** we saw earlier, now expressed in Laplace language.

In [ ]:
alpha = 0.12

roots = np.array([alpha, -alpha])

plt.scatter(roots.real, np.zeros_like(roots))
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")

plt.xlabel("Real part of s")
plt.ylabel("Imaginary part of s")
plt.title("Stable and unstable modes in the s-plane")
plt.grid(True)
plt.show()

# 17. The $s$-plane

The complex plane used for $s$ is called the **$s$-plane**.

A point:

$$
s=\sigma+j\omega
$$

corresponds to a mode:

$$
e^{st}
=
e^{\sigma t}e^{j\omega t}.
$$

Therefore:

### Left half-plane

$$
\sigma<0
$$

means exponential decay.

### Right half-plane

$$
\sigma>0
$$

means exponential growth.

### Imaginary axis

$$
\sigma=0
$$

means undamped oscillation in the ideal linear model.

# 18. Visualize several $s$-plane modes

In [ ]:
modes = [
    (-0.5, 0.0, "decay"),
    (0.3, 0.0, "growth"),
    (-0.2, 2.0, "damped oscillation"),
    (0.0, 2.0, "undamped oscillation"),
]

tt = np.linspace(0, 15, 1000)

for sigma, omega, label in modes:
    y = np.exp(sigma * tt) * np.cos(omega * tt)

    plt.figure()
    plt.plot(tt, y)
    plt.xlabel("Time [s]")
    plt.ylabel("Mode amplitude")
    plt.title(f"{label}: s = {sigma:+.1f} + j{omega:.1f}")
    plt.grid(True)
    plt.show()

The $s$-plane is therefore not an abstract trick.

It is a compact map of:

- decay;
- growth;
- oscillation;
- damping.

This is why classical control engineering is built around it.

# 19. Laplace transform and convolution

A linear dynamical system has memory.

Its output depends on the past history of the input.

In the time domain, this often appears as a convolution:

$$
y(t)
=
\int_0^t h(\tau)u(t-\tau)\,d\tau.
$$

Symbolically:

$$
y=h*u.
$$

Laplace transforms convert convolution into multiplication:

$$
\boxed{
Y(s)=H(s)U(s)
}
$$

This is another major reason they are so powerful.

# 20. Why this matters for control

Suppose the plant is represented by:

$$
Y(s)=G(s)U(s).
$$

And a controller by:

$$
U(s)=C(s)E(s).
$$

Instead of manipulating coupled differential equations, we can manipulate algebraic blocks:

```text
reference
   |
   v
 error ---> C(s) ---> G(s) ---> output
   ^                     |
   |_____________________|
```

This will become the language of transfer functions and feedback systems.

# 21. Step input as a diagnostic experiment

The unit step:

$$
u(t)=1
$$

has transform:

$$
U(s)=\frac{1}{s}.
$$

Why is the step so common in control engineering?

Because it asks:

> What does the system do after a sudden persistent change in command?

The resulting **step response** reveals:

- speed;
- overshoot;
- oscillation;
- settling;
- steady-state error.

# 22. Impulse input

An ideal impulse is written:

$$
\delta(t).
$$

Its Laplace transform is:

$$
\mathcal{L}\{\delta(t)\}=1.
$$

Therefore, if:

$$
Y(s)=G(s)U(s),
$$

an impulse input gives:

$$
Y(s)=G(s).
$$

The inverse transform of $G(s)$ is the system's **impulse response**.

This will be important in Notebook 15.

# 23. A numerical check

Consider:

$$
\dot x+x=1,
\qquad x(0)=0.
$$

Laplace analysis predicts:

$$
x(t)=1-e^{-t}.
$$

Let's compare this with direct numerical integration.

In [ ]:
dt = 0.01
tt = np.arange(0, 6 + dt, dt)

x_num = np.zeros_like(tt)

for k in range(len(tt)-1):
    dx = 1.0 - x_num[k]
    x_num[k+1] = x_num[k] + dx*dt

x_exact = 1 - np.exp(-tt)

plt.plot(tt, x_num, label="Numerical integration")
plt.plot(tt, x_exact, linestyle="--", label="Laplace solution")

plt.xlabel("Time [s]")
plt.ylabel("x(t)")
plt.title("Two methods, same dynamics")
plt.grid(True)
plt.legend()
plt.show()

The numerical integrator and the Laplace method are not competing descriptions.

They solve or represent the same system in different ways.

- numerical integration is excellent for simulation;
- Laplace analysis is excellent for understanding linear input-output structure.

# 24. State-space vs Laplace-domain thinking

We now have two complementary languages.

## State space

$$
\dot x=Ax+Bu
$$

focuses on the internal state.

It is especially powerful for:

- multivariable systems;
- state estimation;
- state feedback;
- optimal control.

## Laplace / transfer-function approach

$$
Y(s)=G(s)U(s)
$$

focuses on input-output behavior.

It is especially powerful for:

- classical feedback analysis;
- poles and zeros;
- PID;
- frequency response;
- Bode plots.

DiveLab will use both.

# 25. A preview of transfer functions

From:

$$
\dot x+2x=u,
$$

with zero initial conditions:

$$
(s+2)X=U.
$$

Therefore:

$$
\frac{X(s)}{U(s)}
=
\frac{1}{s+2}.
$$

The ratio:

$$
\boxed{
G(s)=\frac{X(s)}{U(s)}
}
$$

is called a **transfer function**.

That is the subject of Notebook 15.

# Exercises

### 1. Transform an exponential

Use SymPy to compute:

$$
\mathcal{L}\{e^{-3t}\}.
$$

Identify the corresponding location in the $s$-plane.

In [ ]:
# Your code here

### 2. Solve a first-order system

Solve using Laplace transforms:

$$
\dot x+4x=2,
\qquad x(0)=0.
$$

Then verify the result numerically.

In [ ]:
# Your code here

### 3. Nonzero initial condition

Solve:

$$
\dot x+2x=0,
\qquad x(0)=3.
$$

What part of the Laplace derivative formula carries the initial condition?

### 4. Second-order roots

For:

$$
\ddot x+4\dot x+3x=0,
$$

find the roots of:

$$
s^2+4s+3=0.
$$

Predict the qualitative time behavior before solving for $x(t)$.

In [ ]:
# Your code here

### 5. Stable or unstable?

Consider:

$$
s^2-0.04=0.
$$

Find both roots.

Where are they in the $s$-plane?

What does this imply about the equilibrium?

# Challenge — connect state space and Laplace

Consider:

$$
\dot x=Ax+Bu
$$

with:

$$
A=
\begin{bmatrix}
0 & -1\\
-0.04 & 0
\end{bmatrix}.
$$

1. Compute the eigenvalues of $A$.
2. Derive the characteristic polynomial:

$$
\det(sI-A).
$$

3. Find its roots.
4. Compare them with the eigenvalues.

You should discover that the two analyses describe the same modes.

In [ ]:
A = np.array([
    [0.0, -1.0],
    [-0.04, 0.0]
])

print("Eigenvalues:")
print(np.linalg.eigvals(A))

# Continue here...

# Summary

In this notebook we introduced the Laplace transform as a new language for linear dynamical systems.

We learned that:

- the transform maps $f(t)$ to $F(s)$;
- $s=\sigma+j\omega$ combines growth/decay and oscillation;
- derivatives become algebraic expressions in $s$;
- initial conditions appear explicitly in derivative transforms;
- differential equations can become algebraic equations;
- denominator roots correspond to natural modes;
- right-half-plane modes grow;
- left-half-plane modes decay;
- eigenvalues and Laplace-domain poles are closely connected;
- convolution becomes multiplication;
- the method naturally leads to transfer functions.

### Core insight

$$
\boxed{
\text{Differential equations in }t
\quad\longrightarrow\quad
\text{algebraic equations in }s
}
$$

The Laplace transform does not remove dynamics.

It **encodes dynamics algebraically**.

### Next — Notebook 15

We will define and study **transfer functions**:

$$
G(s)=\frac{Y(s)}{U(s)}.
$$

Then we can make the connection:

$$
\boxed{
\text{differential equation}
\rightarrow
\text{Laplace transform}
\rightarrow
\text{transfer function}
\rightarrow
\text{poles and zeros}
\rightarrow
\text{system behavior}
}
$$